In [14]:
# ================================
# IMPORTS + UPLOAD DO CLEAN CSV
# ================================
import os
import pandas as pd

from google.colab import files

# Se o CSV clean não estiver no runtime, pedir upload
if not os.path.exists("revolut_reviews_clean.csv"):
    print("⬆️ Faça upload do arquivo: revolut_reviews_clean.csv")
    files.upload()

print("✅ Arquivos no runtime:", os.listdir("."))

✅ Arquivos no runtime: ['.config', 'revolut_reviews.csv', 'revolut_reviews_clean.csv', 'sample_data']


In [15]:
# ================================
# LOAD + VALIDAÇÃO
# ================================
df = pd.read_csv("revolut_reviews_clean.csv")

required_cols = ["rating", "review_text", "review_len"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise Exception(f"❌ CSV clean está faltando colunas: {missing}. Colunas existentes: {df.columns.tolist()}")

df.head(), df.shape

(   rating                                        review_text  review_len  \
 0       5  good bank but kept trying to change my current...         259   
 1       5                                great, easy to use.          19   
 2       5                                               good           4   
 3       5  i think it's a really handy app to have, i was...         179   
 4       1  i was doing an order before midnight to avoid ...         227   
 
    churn_risk  
 0           0  
 1           0  
 2           0  
 3           0  
 4           1  ,
 (1200, 4))

In [17]:
# ================================
# FEATURE: SENTIMENT (TextBlob)
# ================================
from textblob import TextBlob

# Se já existir, não recalcula
if "sentiment_polarity" not in df.columns:
    df["sentiment_polarity"] = df["review_text"].astype(str).apply(
        lambda x: float(TextBlob(x).sentiment.polarity)
    )

df[["rating", "review_len", "sentiment_polarity"]].head()

,rating,review_len,sentiment_polarity
0,5,259,0.366667
1,5,19,0.616667
2,5,4,0.700000
3,5,179,0.300000
4,1,227,0.225000


In [18]:
# ================================
# MODELO: REGRESSÃO PARA PREVER RATING
# ================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df[["sentiment_polarity", "review_len"]]
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("✅ MAE:", mean_absolute_error(y_test, pred))
print("✅ R2:", r2_score(y_test, pred))

✅ MAE: 0.5635910749127238
✅ R2: 0.28872447215803


In [19]:
# ================================
# EXEMPLOS DE PREVISÃO
# ================================
samples = [
    "My account is locked and support is not responding.",
    "Love the app, transfers are fast and easy.",
    "Card keeps getting declined, terrible experience.",
    "Verification fails every time, cannot access my funds."
]

sample_df = pd.DataFrame({
    "review_text": samples,
    "review_len": [len(s) for s in samples],
    "sentiment_polarity": [float(TextBlob(s).sentiment.polarity) for s in samples],
})

sample_df["predicted_rating"] = model.predict(sample_df[["sentiment_polarity", "review_len"]])

sample_df

,review_text,review_len,sentiment_polarity,predicted_rating
0,My account is locked and support is not respon...,51,0.000000,4.057309
1,"Love the app, transfers are fast and easy.",42,0.377778,4.531723
2,"Card keeps getting declined, terrible experience.",49,-1.000000,2.897978
3,"Verification fails every time, cannot access m...",54,-0.500000,3.462700


In [20]:
# ================================
# SALVAR RESULTADOS PARA O README
# ================================
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.2f}")

MAE: 0.56
R2: 0.29


In [21]:
# ================================
# EXPORTAR DATASET COM SENTIMENT (OPCIONAL)
# ================================
df.to_csv("revolut_reviews_clean_with_sentiment.csv", index=False)
files.download("revolut_reviews_clean_with_sentiment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>